In [ ]:
import sys, os, pickle, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, mannwhitneyu, kruskal, fisher_exact
from statsmodels.stats.multitest import multipletests

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
from spk_lfp_cluster_comp_analysis import *
from spk_feat_cluster_comp_analysis import compile_experiment_results
from config import SPE1_PICKLE_ROOT, PRIORITY_CELLS, CELL_IDS, DICT_CELL_TYPE, DICT_PATCH_TYPE, DICT_CORT_DEPTH, DICT_DARK_NEURONS, DICT_EAP_WAV

warnings.filterwarnings("ignore")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
SLIDING_DIR  = os.path.join(SPE1_PICKLE_ROOT, "lfp_spk_group_pickles")
PREPOST_DIR  = os.path.join(SPE1_PICKLE_ROOT, "prepost_specparam_pickles")
CLUSTER_DIR  = os.path.join(SPE1_PICKLE_ROOT, "cluster_pickles")

# ── Cell subsets ───────────────────────────────────────────────────────────────
subsets = get_cell_subsets(PRIORITY_CELLS, CELL_IDS)
SUBSET_LABELS = [
    ("Priority cells",      subsets["priority"]),
    ("Non-priority cells",  subsets["np"]),
    ("All cells combined",  subsets["all"]),
]
print(f"Priority: {len(subsets['priority'])} | NP: {len(subsets['np'])} | All: {len(subsets['all'])}")

# ── Feature type definitions ───────────────────────────────────────────────────
# Numerical LFP features (effect sizes, continuous)
LFP_NUMERICAL = ["exponent", "r_squared", "band_aucs.theta",
                  "band_aucs.gamma", "lfp_mean"]
# Dropped as redundant: offset (collinear w/ exponent), lfp_std, lfp_exponent

# Spike feature colours
feature_shades = {
    'peak_amp_cluster':         '#8c564b',
    'peak_sharpness_cluster':   '#a06d62',
    'peak_width_cluster':       '#b38479',
    'exp_lambda_cluster':       '#c561a8',
    'inflection_time_cluster':  '#9b59b6',
    'exp_const_cluster':        '#d7aee0',
    'log_isi_cluster':          '#7f7f7f',
    'spk_times_ms_cluster':     '#b0b0b0',
}

# Module-level constants (not exported by import *)
_CB_PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7',
               '#56B4E9', '#E69F00', '#F0E442', '#000000']
_SIG_COL    = '#D55E00'
_INSIG_COL  = '#56B4E9'
_FS_SM, _FS_AX, _FS_SUB, _FS_TTL = 11, 13, 14, 16


## Load data

In [ ]:
df_pop_stats, pop_traces = compile_lfp_stats(SLIDING_DIR)
df_summary = summarise_cell_lfp_effects(df_pop_stats)
print(f"{df_pop_stats['cell_id'].nunique()} cells | {len(df_pop_stats)} windows | "
      f"{df_pop_stats['lfp_feature'].nunique()} LFP features | "
      f"{df_pop_stats['spike_feature'].nunique()} spike features")

## LFP feature redundancy

Before anything else — do we actually need all LFP features? We compute pairwise Spearman ρ between features using max |Cohen's d| across cells. High ρ = redundant. We drop `offset`, `lfp_std`, and `lfp_exponent` based on known collinearity and keep the reduced set (`exponent`, `r²`, `theta_auc`, `gamma_auc`, `lfp_mean`).

In [ ]:
corr_mat = compute_lfp_feature_correlations(df_pop_stats)
plot_lfp_feature_correlation_matrix(corr_mat, "LFP Feature Redundancy — Sliding Window")

# Filter to reduced feature set for all downstream analyses
df_reduced = df_pop_stats[df_pop_stats["lfp_feature"].isin(LFP_NUMERICAL)].copy()
df_sum_red = summarise_cell_lfp_effects(df_reduced)
print(f"Keeping {df_reduced['lfp_feature'].nunique()} features: {sorted(df_reduced['lfp_feature'].unique())}")

## Effect sizes across the population

Distribution of Cohen's d (significant windows only) for each LFP × spike feature pair, split by cell subset. Boxplot = population spread; dots = individual cells.

In [ ]:
for label, cell_ids in SUBSET_LABELS:
    print(f"\n{'='*60}\n{label} (n={len(cell_ids)})\n{'='*60}")
    df_sub = filter_df_stats(df_reduced, cell_ids)
    if df_sub.empty: print("  No data."); continue
    plot_population_effect_sizes(df_sub, feature_shades)

## When do LFP effects occur relative to the spike?

Density of significant windows as a function of time relative to spike peak. Tells us whether LFP modulation is pre-spike, post-spike, or peri-spike.

In [ ]:
for label, cell_ids in SUBSET_LABELS:
    print(f"\n{'='*60}\n{label} (n={len(cell_ids)})\n{'='*60}")
    df_sub = filter_df_stats(df_reduced, cell_ids)
    if df_sub.empty: continue
    plot_temporal_significance_density(df_sub, feature_shades)

## How consistent are effects across cells?

What fraction of cells show a significant effect for each LFP × spike feature combination?

In [ ]:
for label, cell_ids in SUBSET_LABELS:
    print(f"\n{'='*60}\n{label} (n={len(cell_ids)})\n{'='*60}")
    df_sub     = filter_df_stats(df_reduced, cell_ids)
    traces_sub = filter_traces(pop_traces, cell_ids)
    if df_sub.empty: continue
    plot_significant_yield_heatmap(df_sub, traces_sub)

## Population grand average traces

Mean LFP trace per cluster group (low vs high) averaged across cells that showed a significant effect.

In [ ]:
for label, cell_ids in SUBSET_LABELS:
    print(f"\n{'='*60}\n{label} (n={len(cell_ids)})\n{'='*60}")
    df_sub     = filter_df_stats(df_reduced, cell_ids)
    traces_sub = filter_traces(pop_traces, cell_ids)
    if df_sub.empty: continue
    plot_all_grand_average_traces(traces_sub, df_sub)

## Validation

Bootstrap test: are effects stable across the cell population? Binomial test: is the yield above chance?

In [ ]:
for label, cell_ids in SUBSET_LABELS:
    print(f"\n{'='*60}\n{label} (n={len(cell_ids)})\n{'='*60}")
    df_sub     = filter_df_stats(df_reduced, cell_ids)
    traces_sub = filter_traces(pop_traces, cell_ids)
    if df_sub.empty: continue
    boot_results  = bootstrap_population_stats(df_sub, master_traces=traces_sub)
    binom_results = binomial_yield_test(df_sub, master_traces=traces_sub)

## Conclusions

Validated LFP × spike feature combos with direction and timing summary.

In [ ]:
for label, cell_ids in SUBSET_LABELS:
    print(f"\n{'='*60}\n{label} (n={len(cell_ids)})\n{'='*60}")
    df_sub     = filter_df_stats(df_reduced, cell_ids)
    traces_sub = filter_traces(pop_traces, cell_ids)
    if df_sub.empty: continue
    boot_results  = bootstrap_population_stats(df_sub, master_traces=traces_sub)
    binom_results = binomial_yield_test(df_sub, master_traces=traces_sub)
    df_conclusions = summarize_significant_combos(
        df_stats=df_sub, master_traces=traces_sub,
        boot_results=boot_results, perm_summary=None)
    plot_validated_directions(df_sub, traces_sub, df_conclusions=df_conclusions)